In [ ]:
import os
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from openpyxl import load_workbook
from openpyxl.styles import Alignment, PatternFill
from openpyxl.utils import get_column_letter
import platform
import subprocess
from webdriver_manager.chrome import ChromeDriverManager



# 配置 Chrome 浏览器选项
options = Options()
options.add_argument(r"--user-data-dir=C:\Users\12082\AppData\Local\Google\Chrome\SeleniumData")  # 替换为实际用户数据目录
options.add_argument("--start-maximized")  # 最大化窗口

# 创建 Chrome WebDriver
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# 打开目标页面
url = "https://member.bilibili.com/platform/upload-manager/article"
driver.get(url)

# 等待页面加载，第一次运行需要登录账号
time.sleep(5)

# 用于保存所有数据
all_video_data = []
click_count = 0  # 记录点击次数

try:
    while True:
        # 等待页面加载视频数据
        time.sleep(3)

        # 定位每篇文章的容器元素
        articles = driver.find_elements(By.CSS_SELECTOR, "div.article-card.clearfix.v2")
        
        for article in articles:
            try:
                
                # 提取文章标题
                title_element = article.find_element(By.CSS_SELECTOR, "a.name.ellipsis")
                title = title_element.text.strip()

                # 提取日期
                date_element = article.find_element(By.CSS_SELECTOR, "span.date")
                raw_date = date_element.text.strip()
                print(f"提取到的原始日期: {raw_date}")  # 调试输出

                # 处理日期，只取到“日”
                if " " in raw_date:
                    date = raw_date.split(" ")[0]
                else:
                    date = raw_date

                print(f"截取后的日期: {date}")  # 调试输出

                # 提取播放量、点赞量、评论量、投币量、收藏量、转发量
                stats = article.find_elements(By.CSS_SELECTOR, "span.icon-text")
                play_count = stats[0].text.strip() if len(stats) > 0 else "0"
                like_count = stats[1].text.strip() if len(stats) > 1 else "0"
                comment_count = stats[2].text.strip() if len(stats) > 2 else "0"
                coin_count = stats[3].text.strip() if len(stats) > 3 else "0"
                favorite_count = stats[4].text.strip() if len(stats) > 4 else "0"
                share_count = stats[5].text.strip() if len(stats) > 5 else "0"

                # 保存到数据列表
                all_video_data.append({
                    "标题": title,
                    "日期": date,
                    "播放量": int(play_count.replace(",", "")),  # 转为整数，去掉逗号
                    "点赞量": int(like_count.replace(",", "")),
                    "评论量": int(comment_count.replace(",", "")),
                    "投币量": int(coin_count.replace(",", "")),
                    "收藏量": int(favorite_count.replace(",", "")),
                    "转发量": int(share_count.replace(",", ""))
                })

            except Exception as e:
                print(f"提取数据失败: {e}")

        # 尝试定位“下一页”按钮
        try:
            next_button = driver.find_element(By.CSS_SELECTOR, "li.bcc-pagination-item.bcc-pagination-next")

            # 检查按钮是否为禁用状态
            if "disabled" in next_button.get_attribute("class"):
                print("已到最后一页，停止翻页")
                break

            # 点击“下一页”按钮
            next_button.click()
            print("点击下一页按钮...")

        except Exception as e:
            print("下一页按钮不可用或定位失败:", e)
            break

except Exception as e:
    print("提取过程中出现错误:", e)

finally:
    # 关闭浏览器
    driver.quit()

# 将数据保存到 Excel 文件
desktop = os.path.join(os.path.expanduser("~"), "Desktop")  # 获取桌面路径
output_file = os.path.join(desktop, "B站视频数据.xlsx")  # Excel 文件路径

# 将数据保存到 DataFrame
df = pd.DataFrame(all_video_data)

# 保存到 Excel
df.to_excel(output_file, index=False, sheet_name="视频数据")

# 调整 Excel 文件的样式
wb = load_workbook(output_file)
ws = wb.active

# 设置标题列宽度为 40 像素
ws.column_dimensions[get_column_letter(1)].width = 40  # 第一列 (标题列)

# 设置日期列宽度为 20 像素，右对齐
ws.column_dimensions[get_column_letter(2)].width = 20  # 日期列
alignment_right = Alignment(horizontal="right")
for row in ws.iter_rows(min_row=2, min_col=2, max_col=2, max_row=ws.max_row):  # 日期列右对齐
    for cell in row:
        cell.alignment = alignment_right

# 设置其他列宽度为 10 像素，右对齐
for col in range(3, len(df.columns) + 1):
    ws.column_dimensions[get_column_letter(col)].width = 10
    for row in ws.iter_rows(min_row=2, min_col=col, max_col=col, max_row=ws.max_row):
        for cell in row:
            cell.alignment = alignment_right

# 找到播放量最大值并填充黄色
max_play_count = df["播放量"].max()  # 找到最大播放量
for row in ws.iter_rows(min_row=2, min_col=3, max_col=3, max_row=ws.max_row):
    for cell in row:
        if cell.value == max_play_count:
            cell.fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")  # 填充黄色

# 保存调整后的 Excel 文件
wb.save(output_file)
print(f"数据已保存到 Excel 文件: {output_file}")

# 自动打开 Excel 文件
try:
    system_platform = platform.system()
    if system_platform == "Darwin":  # macOS
        subprocess.run(["open", output_file], check=True)
    elif system_platform == "Windows":  # Windows
        os.startfile(output_file)
    elif system_platform == "Linux":  # Linux
        subprocess.run(["xdg-open", output_file], check=True)
    else:
        print("无法识别操作系统，无法自动打开文件。")
except Exception as e:
    print(f"无法自动打开文件: {e}")
